In [1]:
# ## NYC Taxi Data Ingest Notebook
# This notebook will download NYC yellow taxi CSV data and load it into a PostgreSQL database.

# ### 1️⃣ Install required packages (run this cell if needed)
!pip install --quiet pandas sqlalchemy psycopg2-binary tqdm click


In [2]:
# ### 2️⃣ Imports
import pandas as pd
from sqlalchemy import create_engine
from tqdm.auto import tqdm


In [3]:
# ### 3️⃣ Define CSV URL and dtypes
year = 2021
month = 1

prefix = 'https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/'
url = f'{prefix}yellow_tripdata_{year}-{month:02d}.csv.gz'
print("CSV URL:", url)

dtype = {
    "VendorID": "Int64",
    "passenger_count": "Int64",
    "trip_distance": "float64",
    "RatecodeID": "Int64",
    "store_and_fwd_flag": "string",
    "PULocationID": "Int64",
    "DOLocationID": "Int64",
    "payment_type": "Int64",
    "fare_amount": "float64",
    "extra": "float64",
    "mta_tax": "float64",
    "tip_amount": "float64",
    "tolls_amount": "float64",
    "improvement_surcharge": "float64",
    "total_amount": "float64",
    "congestion_surcharge": "float64"
}

parse_dates = ["tpep_pickup_datetime", "tpep_dropoff_datetime"]


CSV URL: https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2021-01.csv.gz


In [7]:
# ### 4️⃣ Connect to PostgreSQL
# Use 'pgdatabase' if running inside Docker container network
# Use 'localhost' if connecting from your Mac terminal to port 5432

db_user = 'root'
db_pass = 'root'
db_host = 'localhost'  # change to 'localhost' if running outside Docker
db_port = '5432'
db_name = 'ny_taxi'

engine = create_engine(f'postgresql://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}')


In [8]:
# ### 6️⃣ Load CSV in chunks to PostgreSQL
chunksize = 100_000
df_iter = pd.read_csv(url, dtype=dtype, parse_dates=parse_dates, iterator=True, chunksize=chunksize)

for i, df_chunk in enumerate(tqdm(df_iter, desc="Inserting chunks")):
    df_chunk.to_sql(name='yellow_taxi_data', con=engine, if_exists='append')
    print(f"Chunk {i+1} inserted")


Inserting chunks: 0it [00:00, ?it/s]

Chunk 1 inserted
Chunk 2 inserted
Chunk 3 inserted
Chunk 4 inserted
Chunk 5 inserted
Chunk 6 inserted
Chunk 7 inserted
Chunk 8 inserted
Chunk 9 inserted
Chunk 10 inserted
Chunk 11 inserted
Chunk 12 inserted
Chunk 13 inserted
Chunk 14 inserted


In [9]:
# ### 7️⃣ Load taxi zone lookup (optional)
df_zones = pd.read_csv('taxi_zone_lookup.csv')
df_zones.to_sql(name='zones', con=engine, if_exists='replace')
df_zones.head()


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [10]:
# ### ✅ Done
print("Data has been ingested into PostgreSQL!")


Data has been ingested into PostgreSQL!
